# Stagbation metric fetching
This notebook aims to fetch stagnation signals from MD&A sections of 10-K filings from S&P 500 companies through 4 NLP metrics, calculating stagnation NLP score. 
Along with an initial version of financial metrics (a newer version of financial metric fetching is included in financial_metric_fetch.ipynb)

In [47]:
# !pip install yfinance
# !pip install edgartools
import time
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import pprint
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
from edgar import *
import edgar

## Financial data collection from yFinance

In [49]:
def get_sp500_tickers_bs4():
    """
    Fetch S&P 500 tickers from Wikipedia using BeautifulSoup with proper headers.
    
    Returns:
        list: List of ticker symbols (e.g., ['MMM', 'AOS', ...])
    """
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    
    # Mimic a real browser to avoid 403 errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find the first wikitable (S&P 500 constituents)
        table = soup.find('table', {'class': 'wikitable'})
        if not table:
            print("Could not find the table.")
            return []
        
        tickers = []
        # Skip header row (first <tr>)
        for row in table.find_all('tr')[1:]:
            cells = row.find_all('td')
            if cells:
                ticker = cells[0].get_text(strip=True)
                tickers.append(ticker)
        
        return tickers
    
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []

if __name__ == "__main__":
    tickers = get_sp500_tickers_bs4()
    print(f"Retrieved {len(tickers)} tickers.")
    print("First 10 tickers:", tickers[:10])

Retrieved 503 tickers.
First 10 tickers: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [50]:

def fetch_financials_from_yfinance(ticker, years=4):
    """
    Fetches comprehensive financial data for a ticker for the last 'years' fiscal years.
    Returns a dictionary: year -> metrics dict.
    """
    try:
        stock = yf.Ticker(ticker)
        
        # Get annual statements, limit to last 'years' years
        income = stock.income_stmt.iloc[:, :years]          # columns are fiscal year ends
        balance = stock.balance_sheet.iloc[:, :years]
        cashflow = stock.cashflow.iloc[:, :years]
        
        if income.empty:
            return {}
        
        # Company info
        info = stock.info
        sector = info.get('sector')
        industry = info.get('industry')
        shares_outstanding = info.get('sharesOutstanding')
        current_market_cap = info.get('marketCap')
        
        results = []
        # Iterate over columns (fiscal year ends)
        for col in income.columns:
            year = col.year
            # Ensure same column exists in all statements
            if col not in balance.columns or col not in cashflow.columns:
                continue
            
            # Income statement
            revenue = income.loc['Total Revenue'].get(col) if 'Total Revenue' in income.index else income.loc['Revenue'].get(col) if 'Revenue' in income.index else None
            ebitda = income.loc['EBITDA'].get(col) if 'EBITDA' in income.index else None
            ebit = income.loc['EBIT'].get(col) if 'EBIT' in income.index else income.loc['Operating Income'].get(col) if 'Operating Income' in income.index else None
            net_income = income.loc['Net Income'].get(col) if 'Net Income' in income.index else None
            interest_expense = income.loc['Interest Expense'].get(col) if 'Interest Expense' in income.index else None
            
            # Balance sheet
            total_assets = balance.loc['Total Assets'].get(col) if 'Total Assets' in balance.index else None
            total_liabilities = balance.loc['Total Liabilities'].get(col) if 'Total Liabilities' in balance.index else None
            shareholders_equity = balance.loc['Stockholders Equity'].get(col) if 'Stockholders Equity' in balance.index else None
            current_assets = balance.loc['Current Assets'].get(col) if 'Current Assets' in balance.index else None
            current_liabilities = balance.loc['Current Liabilities'].get(col) if 'Current Liabilities' in balance.index else None
            short_term_debt = balance.loc['Short Term Debt'].get(col) if 'Short Term Debt' in balance.index else 0
            long_term_debt = balance.loc['Long Term Debt'].get(col) if 'Long Term Debt' in balance.index else 0
            cash_equiv = balance.loc['Cash And Cash Equivalents'].get(col) if 'Cash And Cash Equivalents' in balance.index else 0
            
            total_debt = short_term_debt + long_term_debt
            
            # Cash flow
            operating_cf = cashflow.loc['Operating Cash Flow'].get(col) if 'Operating Cash Flow' in cashflow.index else None
            capex = cashflow.loc['Capital Expenditure'].get(col) if 'Capital Expenditure' in cashflow.index else None
            fcf = (operating_cf + capex) if operating_cf is not None and capex is not None else None
            
            # Use current market cap for all years (simplification)
            market_cap = current_market_cap
            
            # Enterprise Value
            ev = None
            if market_cap and total_debt is not None and cash_equiv is not None:
                ev = market_cap + total_debt - cash_equiv
            
            # Ratios
            ebitda_margin = (ebitda / revenue) if ebitda and revenue and revenue != 0 else None
            fcf_margin = (fcf / revenue) if fcf and revenue and revenue != 0 else None
            debt_to_ebitda = (total_debt / ebitda) if total_debt and ebitda and ebitda != 0 else None
            interest_coverage = (ebit / interest_expense) if ebit and interest_expense and interest_expense != 0 else None
            current_ratio = (current_assets / current_liabilities) if current_assets and current_liabilities and current_liabilities != 0 else None
            roa = (net_income / total_assets) if net_income and total_assets and total_assets != 0 else None
            roe = (net_income / shareholders_equity) if net_income and shareholders_equity and shareholders_equity != 0 else None
            ev_to_ebitda = (ev / ebitda) if ev and ebitda and ebitda != 0 else None
            price_to_sales = (market_cap / revenue) if market_cap and revenue and revenue != 0 else None
            
            metrics = {
                'ticker': ticker,
                'year': year,
                'fiscal_year_end': col,
                'sector': sector,
                'industry': industry,
                'revenue': revenue,
                'ebitda': ebitda,
                'ebitda_margin': ebitda_margin,
                'net_income': net_income,
                'total_assets': total_assets,
                'total_liabilities': total_liabilities,
                'shareholders_equity': shareholders_equity,
                'total_debt': total_debt,
                'cash_equiv': cash_equiv,
                'operating_cf': operating_cf,
                'capex': capex,
                'fcf': fcf,
                'fcf_margin': fcf_margin,
                'debt_to_ebitda': debt_to_ebitda,
                'interest_coverage': interest_coverage,
                'current_ratio': current_ratio,
                'roa': roa,
                'roe': roe,
                'market_cap': market_cap,
                'enterprise_value': ev,
                'ev_to_ebitda': ev_to_ebitda,
                'price_to_sales': price_to_sales
            }
            results.append(metrics)
        
        # Sort by year ascending and add growth metrics
        results.sort(key=lambda x: x['year'])
        for i in range(1, len(results)):
            prev = results[i-1]
            curr = results[i]
            curr['revenue_growth'] = (curr['revenue'] - prev['revenue']) / abs(prev['revenue']) if prev['revenue'] and curr['revenue'] and prev['revenue'] != 0 else None
            curr['ebitda_growth'] = (curr['ebitda'] - prev['ebitda']) / abs(prev['ebitda']) if prev['ebitda'] and curr['ebitda'] and prev['ebitda'] != 0 else None
            curr['fcf_growth'] = (curr['fcf'] - prev['fcf']) / abs(prev['fcf']) if prev['fcf'] and curr['fcf'] and prev['fcf'] != 0 else None
        
        # Convert to dict by year
        results_dict = {m['year']: m for m in results}
        return results_dict
    
    except Exception as e:
        print(f"Error fetching financials for {ticker}: {e}")
        return {}

def add_growth_metrics(results):
    """Adds YoY growth rates for revenue, EBITDA, etc."""
    if len(results) < 2:
        return results
    # Sort by year ascending
    results.sort(key=lambda x: x['year'])
    for i in range(1, len(results)):
        prev = results[i-1]
        curr = results[i]
        # Revenue growth
        if prev['revenue'] and curr['revenue'] and prev['revenue'] != 0:
            curr['revenue_growth'] = (curr['revenue'] - prev['revenue']) / abs(prev['revenue'])
        else:
            curr['revenue_growth'] = None
        # EBITDA growth
        if prev['ebitda'] and curr['ebitda'] and prev['ebitda'] != 0:
            curr['ebitda_growth'] = (curr['ebitda'] - prev['ebitda']) / abs(prev['ebitda'])
        else:
            curr['ebitda_growth'] = None
        # FCF growth
        if prev['fcf'] and curr['fcf'] and prev['fcf'] != 0:
            curr['fcf_growth'] = (curr['fcf'] - prev['fcf']) / abs(prev['fcf'])
        else:
            curr['fcf_growth'] = None
    return results

def get_capex_by_year(ticker_data):
    capex_by_year = {year: data['capex'] for year, data in ticker_data.items() if data['capex'] is not None}
    return capex_by_year


def assess_financial_feasibility(ticker, years=4, target_interest_coverage=3.0):
    """
    Assesses financial feasibility using fetched yfinance data.
    Returns a dict with metrics and qualitative ratings for:
      - Debt capacity
      - Cash flow health
      - Liquidity
      - Leverage tolerance
    """
    data = fetch_financials_from_yfinance(ticker, years)
    if not data:
        print(f"No data available for {ticker}")
        return None
    
    # Use the most recent year's data
    latest_year = max(data.keys())
    latest = data[latest_year]
    
    # -----------------------------------------------------------------
    # Helper rating function
    # -----------------------------------------------------------------
    def rate_metric(value, thresholds):
        """thresholds: list of (upper_bound, rating) from worst to best"""
        if value is None:
            return "N/A"
        for bound, rating in thresholds:
            if value <= bound:
                return rating
        return thresholds[-1][1]  # fallback
    
    # 1. LEVERAGE TOLERANCE & DEBT CAPACITY
    #    Interest coverage ratio (EBIT / Interest)
    ic = latest.get('interest_coverage')
    ic_rating = rate_metric(ic, [
        (0.0, "Critical (default risk)"),
        (1.0, "Very Weak"),
        (1.5, "Weak"),
        (2.0, "Marginal"),
        (3.0, "Adequate"),
        (5.0, "Strong"),
        (float('inf'), "Very Strong")
    ])
    
    #    Debt / EBITDA
    d_ebitda = latest.get('debt_to_ebitda')
    d_ebitda_rating = rate_metric(d_ebitda, [
        (0.0, "No debt"),
        (1.0, "Conservative"),
        (2.0, "Moderate"),
        (3.0, "Elevated"),
        (4.0, "High"),
        (5.0, "Very High"),
        (float('inf'), "Distressed")
    ])
    
    #    Debt capacity: how much additional debt can be added while keeping IC >= target_coverage?
    ebit = latest.get('ebit')
    interest_exp = None
    # Need interest expense from income statement – we didn't store it directly, but can recompute from IC if ebit known
    # Better: we should fetch and store interest_exp in fetch function. Let's add it now (quick patch)
    # For demonstration, we'll assume it's available; in practice, modify fetch to include 'interest_expense'
    # Since we have interest_coverage = ebit / interest_exp, we can derive.
    if ic and ebit and ic != 0:
        interest_exp = ebit / ic
        avg_interest_rate = interest_exp / latest.get('total_debt') if latest.get('total_debt') and latest.get('total_debt') != 0 else 0.05
        max_interest = ebit / target_interest_coverage
        additional_interest_capacity = max_interest - interest_exp
        if additional_interest_capacity > 0 and avg_interest_rate > 0:
            additional_debt_capacity = additional_interest_capacity / avg_interest_rate
        else:
            additional_debt_capacity = 0
    else:
        additional_debt_capacity = None
    
    # 2. CASH FLOW HEALTH
    fcf = latest.get('fcf')
    fcf_margin = latest.get('fcf_margin')
    ocf = latest.get('operating_cf')
    # FCF margin rating
    fcf_margin_rating = rate_metric(fcf_margin, [
        (-float('inf'), "Negative"),
        (0.0, "Breakeven"),
        (0.05, "Weak"),
        (0.10, "Moderate"),
        (0.15, "Good"),
        (0.20, "Excellent"),
        (float('inf'), "Exceptional")
    ])
    # Operating cash flow vs. total debt (repayment ability)
    ocf_to_debt = (ocf / latest.get('total_debt')) if ocf and latest.get('total_debt') and latest.get('total_debt') != 0 else None
    ocf_to_debt_rating = rate_metric(ocf_to_debt, [
        (0.0, "Cannot cover interest"),
        (0.05, "Weak"),
        (0.10, "Adequate"),
        (0.20, "Strong"),
        (float('inf'), "Very Strong")
    ])
    
    # 3. LIQUIDITY
    cr = latest.get('current_ratio')
    cr_rating = rate_metric(cr, [
        (0.5, "Critical"),
        (0.8, "Weak"),
        (1.0, "Marginal"),
        (1.2, "Adequate"),
        (1.5, "Good"),
        (2.0, "Strong"),
        (float('inf'), "Very Strong")
    ])
    # Quick ratio (we don't have inventory directly in fetch, but can approximate if needed)
    # For simplicity, we use current ratio as primary liquidity metric.
    
    # 4. GROWTH & SUSTAINABILITY (additional context)
    revenue_growth = latest.get('revenue_growth')
    ebitda_growth = latest.get('ebitda_growth')
    
    # Build assessment
    assessment = {
        'ticker': ticker,
        'latest_year': latest_year,
        'leverage_tolerance': {
            'interest_coverage': ic,
            'rating': ic_rating,
            'debt_to_ebitda': d_ebitda,
            'debt_ebitda_rating': d_ebitda_rating,
        },
        'debt_capacity': {
            'additional_debt_capacity_usd': additional_debt_capacity,
            'assumptions': f"Maintain interest coverage >= {target_interest_coverage}x"
        },
        'cash_flow_health': {
            'free_cash_flow_usd': fcf,
            'fcf_margin': fcf_margin,
            'fcf_margin_rating': fcf_margin_rating,
            'ocf_to_debt': ocf_to_debt,
            'ocf_to_debt_rating': ocf_to_debt_rating,
        },
        'liquidity': {
            'current_ratio': cr,
            'rating': cr_rating,
            'cash_and_equiv_usd': latest.get('cash_equiv'),
        },
        'growth_indicators': {
            'revenue_growth_yoy': revenue_growth,
            'ebitda_growth_yoy': ebitda_growth,
        },
        'overall_feasibility': None  # to be filled
    }
    
    # Overall assessment (simple heuristic)
    scores = []
    if ic is not None:
        scores.append(1 if ic >= 2.0 else 0)
    if d_ebitda is not None:
        scores.append(1 if d_ebitda <= 3.0 else 0)
    if fcf_margin is not None:
        scores.append(1 if fcf_margin >= 0.10 else 0)
    if cr is not None:
        scores.append(1 if cr >= 1.2 else 0)
    total_score = sum(scores)
    if total_score >= 3:
        overall = "Feasible – strong financial health"
    elif total_score >= 2:
        overall = "Marginally feasible – monitor key risks"
    elif total_score >= 1:
        overall = "Challenging – improvement needed"
    else:
        overall = "Infeasible – high risk of distress"
    assessment['overall_feasibility'] = overall
    
    return assessment
    
# aapl_df = fetch_financials_from_yfinance("AAPL")
# aapl_df_capx_by_year = get_capex_by_year(aapl_df)
# pprint.pprint(aapl_df, indent=4)
# pprint.pprint(aapl_df_capx_by_year)

# Text data from SEC 10-K filling

In [51]:


set_identity("oscarchanly@gmail.com")

headers = {'User-Agent':'Dummy Company oscarchanly@gmail.com','Accept-Encoding':'gzip, deflate','Host':'www.sec.gov'}


def get_n_year_10K(ticker, n = 4): 

    """
    Extract 10 K filings in the last n years
    """

    # get 10-K filing from last n year 
    company = Company(ticker)

    filings_4y = company.get_filings(form="10-K").latest(n) 

   
    # filings_list = [filing for filing in filings_4y]

    # # print summary
    # for filing in filings_list:
    #     print(f"{filing.filing_date}: {filing.form} - {filing.company}")

    return filings_4y

def normalize_text(text):
    """

    clean the text for pattern matching

    """
    # Replace non-breaking spaces
    text = text.replace('\u00A0', ' ')
    # # Replace curly apostrophe and other quotes with ASCII apostrophe
    text = text.replace('’', "'").replace('‘', "'").replace('`', "'")

    return text

def extract_mda_from_filing(filing):

    """
    Extract the MD&A (Item 7) text from a 10-K Filing object.
    """

    # Get the primary document's HTML
    url = filing.filing_url
    response = requests.get(url, headers = headers)
    html_content = response.text
    soup = BeautifulSoup(html_content, 'html.parser')
    text = soup.get_text(separator='\n')

    start_patterns = [
        r"item[ \t]*7\.?[ \t]*management's",
        r"item[ \t]*7\.?[ \t]*–[ \t]*management's",
    ]
    end_patterns = [
        r"item[ \t]*7a\.?[ \t]*quantitative",
        r"item[ \t]*8\.?[ \t]*financial\s+statements",
        r"item[ \t]*8\.?[ \t]*–[ \t]*financial\s+statements"
    ]
    lower_text = normalize_text(text.lower())
    
    start_idx = None
    for pat in start_patterns:
        match = re.search(pat, lower_text)
        if match:
            start_idx = match.start()
            break
    if start_idx is None:
        return None
    
    end_idx = len(text)
    for pat in end_patterns:
        match = re.search(pat, lower_text[start_idx:])
        if match:
            end_idx = start_idx + match.start()
            break
        
    
    mda_text = text[start_idx:end_idx].strip()
    mda_text = re.sub(r'^item\s*7\.?\s*', '', mda_text, flags=re.IGNORECASE).strip()
    return mda_text

# demostration 

filing_list_5y = get_n_year_10K("AAPL")

print(filing_list_5y[0])

# print(extract_mda_from_filing(filing_list_5y[0]))

Filing(company='Apple Inc.', cik=320193, form='10-K', filing_date='2025-10-31', accession_no='0000320193-25-000079')


## Stagnation measurement from text data

In [52]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from scipy.stats import pearsonr


# ------------------------------------------------------------
# 1. Innovation Vocabulary Decay
# ------------------------------------------------------------
forward_words = {
    'anticipate', 'expect', 'believe', 'future', 'plan', 'potential',
    'opportunity', 'goal', 'objective', 'strategy', 'forecast', 'project',
    'outlook', 'target', 'aim', 'seek', 'hope', 'intend', 'look forward'
}

def compute_innovation_decay(mda_by_year):
    """
    Returns:
        freq: dict year -> forward-looking word frequency
        slope: linear slope over years (negative = decay)
    """
    years = sorted(mda_by_year.keys())
    freq = {}
    for year in years:
        text = mda_by_year[year]
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        total = len(words)
        if total == 0:
            freq[year] = 0
            continue
        fwd_count = sum(1 for w in words if w in forward_words)
        freq[year] = fwd_count / total
    # Compute trend
    if len(years) >= 3:
        x = np.array(years)
        y = np.array([freq[y] for y in years])
        slope = np.polyfit(x, y, 1)[0]
    else:
        slope = None
    return freq, slope


# ------------------------------------------------------------
# 2. Topic Rigidity Score (using sentence embeddings)
# ------------------------------------------------------------
# Load a sentence transformer model once
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_topic_rigidity(mda_by_year):
    """
    Returns:
        rigidity: average cosine similarity between consecutive years
        similarities: list of yearly similarities
    """
    mda_years = sorted(mda_by_year.keys())
    if len(mda_years) < 2:
        return None, []
    texts = [mda_by_year[y] for y in mda_years]
    # Encode all texts
    embeddings = embed_model.encode(texts)
    similarities = []
    for i in range(len(mda_years)-1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        similarities.append(sim)
    rigidity = np.mean(similarities) if similarities else None
    return rigidity, similarities

# ------------------------------------------------------------
# 3. Strategic Topic Decay
# ------------------------------------------------------------
strategic_keywords = {
    'innovation', 'research', 'development', 'rd', 'new product',
    'launch', 'expansion', 'market share', 'growth', 'opportunity',
    'investment', 'initiative', 'breakthrough', 'pipeline', 'strategic'
}

def compute_strategic_decay(mda_by_year):
    """
    Returns:
        freq: dict year -> strategic keyword frequency
        slope: linear slope (negative = decay)
    """
    years = sorted(mda_by_year.keys())
    freq = {}
    for year in years:
        text = mda_by_year[year]
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        total = len(words)
        if total == 0:
            freq[year] = 0
            continue
        # Count occurrences of any strategic keyword (simple word match)
        strat_count = sum(1 for w in words if w in strategic_keywords)
        freq[year] = strat_count / total
    if len(years) >= 3:
        x = np.array(years)
        y = np.array([freq[y] for y in years])
        slope = np.polyfit(x, y, 1)[0]
    else:
        slope = None
    return freq, slope

# ------------------------------------------------------------
# 4. Management Confidence Signal
# ------------------------------------------------------------
# Load FinBERT model once
finbert_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

def sentiment_score(text):
    """Return positive sentiment probability (0-1) for text."""
    inputs = finbert_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = finbert_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1).squeeze().tolist()
    # FinBERT outputs [negative, neutral, positive]
    return probs[2]   # positive score

def extract_forward_sentences(text, forward_words=forward_words):
    """
    Extract sentences that contain at least one forward-looking word.
    Simple sentence split using regex.
    """
    # Basic sentence split (period followed by space or newline)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    forward_sentences = []
    for sent in sentences:
        if any(word in sent.lower() for word in forward_words):
            forward_sentences.append(sent)
    return forward_sentences

def compute_management_confidence(mda_by_year, capex_by_year):
    """
    Returns a dict with:
        sentiment_by_year: dict year -> average positive sentiment in forward-looking sentences
        capex_growth: dict year (starting from second year) -> YoY growth
        correlation: Pearson correlation between sentiment (t) and next-year capex growth (t+1)
        mismatch_score: (sentiment_avg - normalized growth) or alternative
    """
    years = sorted(set(mda_by_year.keys()) & set(capex_by_year.keys()))
    if len(years) < 2:
        return None
    # Compute sentiment per year (average over forward-looking sentences)
    sentiment = {}
    for y in years:
        text = mda_by_year[y]
        fwd_sents = extract_forward_sentences(text)
        if fwd_sents:
            # For speed, sample first 2000 chars per sentence (avoid huge text)
            sent_scores = [sentiment_score(s[:2000]) for s in fwd_sents if len(s) > 0]
            sentiment[y] = np.mean(sent_scores) if sent_scores else 0.5
        else:
            sentiment[y] = 0.5   # neutral if no forward sentences
    # Compute CapEx growth
    capex = {y: capex_by_year[y] for y in years}
    capex_growth = {}
    for i in range(1, len(years)):
        prev = years[i-1]
        curr = years[i]
        if capex[prev] and capex[curr] and capex[prev] != 0:
            capex_growth[curr] = (capex[curr] - capex[prev]) / abs(capex[prev])
        else:
            capex_growth[curr] = None
    # Align sentiment with growth: sentiment in year t predicts growth in year t+1
    sent_vals = [sentiment[y] for y in years[:-1]]  # sentiment for years except last
    growth_vals = [capex_growth[y] for y in years[1:]]  # growth for second year onward
    # Remove None pairs
    pairs = [(s, g) for s, g in zip(sent_vals, growth_vals) if g is not None]
    if len(pairs) >= 2:
        s_vals, g_vals = zip(*pairs)
        corr, pval = pearsonr(s_vals, g_vals)
    else:
        corr = None
    # Mismatch: high sentiment but low growth. We'll use negative correlation as indicator.
    return {
        'sentiment_by_year': sentiment,
        'capex_growth': capex_growth,
        'sentiment_growth_correlation': corr
    }

def compute_all_stagnation_metrics(ticker, mda_by_year, capex_by_year):
    """
    mda_by_year: dict year -> MD&A text
    capex_by_year: dict year -> capital expenditure (from financials)
    Returns a dictionary with all metrics.
    """
    metrics = {}

    metrics["ticker"] = ticker

    # 1. Innovation decay
    fwd_freq, decay_slope = compute_innovation_decay(mda_by_year)
    metrics['innovation_freq'] = fwd_freq
    metrics['innovation_decay_slope'] = decay_slope
    # 2. Topic rigidity
    rigidity, yearly_sims = compute_topic_rigidity(mda_by_year)
    metrics['topic_rigidity'] = rigidity
    metrics['topic_similarities'] = yearly_sims
    # 3. Strategic decay
    strat_freq, strat_slope = compute_strategic_decay(mda_by_year)
    metrics['strategic_freq'] = strat_freq
    metrics['strategic_decay_slope'] = strat_slope
    # 4. Management confidence (requires capex)
    if capex_by_year:
        conf = compute_management_confidence(mda_by_year, capex_by_year)
        if conf:
            metrics['sentiment_by_year'] = conf['sentiment_by_year']
            metrics['capex_growth'] = conf['capex_growth']
            metrics['sentiment_growth_correlation'] = conf['sentiment_growth_correlation']
    return metrics

In [53]:
def ticker_metric_pipeline(ticker):
    filing_4y = get_n_year_10K(ticker, n = 4)
    if isinstance(filing_4y, edgar.entity.filings.EntityFilings):
        filing_list_4y = [filing for filing in filing_4y]
        if len(filing_list_4y) < 4:
            return None
    else:
        return None
    mda_by_year = {}
    for filing in filing_list_4y:
        year = filing.filing_date.year
        mda_text = extract_mda_from_filing(filing)
        if mda_text:
            mda_by_year[year] = mda_text
        
    ticker_data = fetch_financials_from_yfinance(ticker)
    if not ticker_data:
        return None
    ticker_data_capx_by_year = get_capex_by_year(ticker_data)
    
    metrics = compute_all_stagnation_metrics(ticker, mda_by_year, ticker_data_capx_by_year)
    return metrics

In [ ]:
sp_sticker = get_sp500_tickers_bs4()
tickers_sample = sp_sticker[:] # Sampling for debug

ticker_metric_list = []
financial_metric_list = []
ticker_times = {}

for idx, ticker in enumerate(tickers_sample, start=1):
    print(f"\nProcessing {idx}/{len(tickers_sample)}: {ticker}")
    start_time = time.perf_counter()

    ticker_metric = ticker_metric_pipeline(ticker)

    if ticker_metric:
        ticker_metric_list.append(ticker_metric)
        financial_metric = assess_financial_feasibility(ticker)
        financial_metric_list.append(financial_metric)

    end_time = time.perf_counter()
    elapsed = end_time - start_time
    ticker_times[ticker] = elapsed

    print(f"  Completed in {elapsed:.2f} seconds")


Processing 1/125: PG
  Completed in 8.28 seconds

Processing 2/125: PGR


C:\Users\Oscar\AppData\Local\Temp\ipykernel_39180\1371189382.py:175: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, pval = pearsonr(s_vals, g_vals)


  Completed in 15.96 seconds

Processing 3/125: PLD
  Completed in 22.98 seconds

Processing 4/125: PRU
  Completed in 32.82 seconds

Processing 5/125: PEG
  Completed in 170.79 seconds

Processing 6/125: PTC
  Completed in 31.44 seconds

Processing 7/125: PSA
  Completed in 10.95 seconds

Processing 8/125: PHM
  Completed in 7.61 seconds

Processing 9/125: PWR
  Completed in 7.85 seconds

Processing 10/125: QCOM
  Completed in 134.95 seconds

Processing 11/125: DGX


Truncated SGML: <DOCUMENT> at offset 83591247 has no matching </DOCUMENT>


  Completed in 13.53 seconds

Processing 12/125: Q
  Completed in 0.42 seconds

Processing 13/125: RL
  Completed in 10.54 seconds

Processing 14/125: RJF
  Completed in 13.33 seconds

Processing 15/125: RTX
  Completed in 64.80 seconds

Processing 16/125: O
  Completed in 11.15 seconds

Processing 17/125: REG
  Completed in 61.79 seconds

Processing 18/125: REGN
  Completed in 35.25 seconds

Processing 19/125: RF
  Completed in 23.15 seconds

Processing 20/125: RSG
  Completed in 12.65 seconds

Processing 21/125: RMD
  Completed in 24.03 seconds

Processing 22/125: RVTY
  Completed in 8.02 seconds

Processing 23/125: HOOD
  Completed in 18.72 seconds

Processing 24/125: ROK


...

  Completed in 6.89 seconds

Processing 71/125: TKO
  Completed in 0.33 seconds

Processing 72/125: TTD
  Completed in 28.83 seconds

Processing 73/125: TSCO
  Completed in 24.37 seconds

Processing 74/125: TT
  Completed in 8.03 seconds

Processing 75/125: TDG
  Completed in 44.81 seconds

Processing 76/125: TRV
  Completed in 12.45 seconds

Processing 77/125: TRMB
  Completed in 18.06 seconds

Processing 78/125: TFC
  Completed in 66.88 seconds

Processing 79/125: TYL
  Completed in 16.59 seconds

Processing 80/125: TSN
  Completed in 61.65 seconds

Processing 81/125: USB
  Completed in 7.31 seconds

Processing 82/125: UBER
  Completed in 35.88 seconds

Processing 83/125: UDR


C:\Users\Oscar\AppData\Local\Temp\ipykernel_39180\1371189382.py:175: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, pval = pearsonr(s_vals, g_vals)


  Completed in 18.74 seconds

Processing 84/125: ULTA
  Completed in 8.19 seconds

Processing 85/125: UNP
  Completed in 7.95 seconds

Processing 86/125: UAL
  Completed in 133.21 seconds

Processing 87/125: UPS
  Completed in 218.64 seconds

Processing 88/125: URI
  Completed in 6.40 seconds

Processing 89/125: UNH
  Completed in 7.85 seconds

Processing 90/125: UHS
  Completed in 157.63 seconds

Processing 91/125: VLO
  Completed in 108.13 seconds

Processing 92/125: VTR
  Completed in 9.62 seconds

Processing 93/125: VLTO
  Completed in 0.34 seconds

Processing 94/125: VRSN
  Completed in 47.09 seconds

Processing 95/125: VRSK
  Completed in 8.43 seconds

Processing 96/125: VZ
  Completed in 69.04 seconds

Processing 97/125: VRTX
  Completed in 9.87 seconds

Processing 98/125: VRT
  Completed in 95.43 seconds

Processing 99/125: VTRS
  Completed in 20.90 seconds

Processing 100/125: VICI
  Completed in 84.67 seconds

Processing 101/125: V
  Completed in 26.93 seconds

Processing 102

C:\Users\Oscar\AppData\Local\Temp\ipykernel_39180\1371189382.py:175: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, pval = pearsonr(s_vals, g_vals)


  Completed in 10.51 seconds

Processing 110/125: WM
  Completed in 7.96 seconds

Processing 111/125: WAT
  Completed in 8.20 seconds

Processing 112/125: WEC
  Completed in 159.81 seconds

Processing 113/125: WFC
  Completed in 8.91 seconds

Processing 114/125: WELL
  Completed in 174.79 seconds

Processing 115/125: WST
  Completed in 26.00 seconds

Processing 116/125: WDC
  Completed in 14.75 seconds

Processing 117/125: WY
  Completed in 15.05 seconds

Processing 118/125: WSM
  Completed in 81.21 seconds

Processing 119/125: WMB
  Completed in 30.40 seconds

Processing 120/125: WTW
  Completed in 56.33 seconds

Processing 121/125: WDAY
  Completed in 35.60 seconds

Processing 122/125: WYNN
  Completed in 29.02 seconds

Processing 123/125: XEL
  Completed in 18.44 seconds

Processing 124/125: XYL
  Completed in 43.82 seconds

Processing 125/125: YUM
  Completed in 11.18 seconds


In [55]:
pprint.pprint(ticker_metric_list)

[{'innovation_decay_slope': None,
  'innovation_freq': {},
  'strategic_decay_slope': None,
  'strategic_freq': {},
  'ticker': 'PG',
  'topic_rigidity': None,
  'topic_similarities': []},
 {'capex_growth': {2024: np.float64(-0.13095238095238096),
                   2025: np.float64(-0.22105263157894736)},
  'innovation_decay_slope': np.float64(0.0),
  'innovation_freq': {2023: 0.0, 2024: 0.0, 2025: 0.0, 2026: 0.0},
  'sentiment_by_year': {2023: 0.5, 2024: 0.5, 2025: 0.5},
  'sentiment_growth_correlation': np.float64(nan),
  'strategic_decay_slope': np.float64(0.0),
  'strategic_freq': {2023: 0.0, 2024: 0.0, 2025: 0.0, 2026: 0.0},
  'ticker': 'PGR',
  'topic_rigidity': np.float32(0.99907666),
  'topic_similarities': [np.float32(0.99910337),
                         np.float32(0.9989146),
                         np.float32(0.999212)]},
 {'innovation_decay_slope': np.float64(-0.00024065572249231217),
  'innovation_freq': {2023: 0.006786507776206827,
                      2024: 0.0059849

In [56]:
pprint.pprint(financial_metric_list)

[{'cash_flow_health': {'fcf_margin': np.float64(0.16662711783968487),
                       'fcf_margin_rating': 'Excellent',
                       'free_cash_flow_usd': np.float64(14044000000.0),
                       'ocf_to_debt': np.float64(0.7128225645129026),
                       'ocf_to_debt_rating': 'Very Strong'},
  'debt_capacity': {'additional_debt_capacity_usd': None,
                    'assumptions': 'Maintain interest coverage >= 3.0x'},
  'growth_indicators': {'ebitda_growth_yoy': np.float64(0.059295013727747765),
                        'revenue_growth_yoy': np.float64(0.0029153131284284676)},
  'latest_year': 2025,
  'leverage_tolerance': {'debt_ebitda_rating': 'Moderate',
                         'debt_to_ebitda': np.float64(1.0448977885539903),
                         'interest_coverage': np.float64(23.2348401323043),
                         'rating': 'Very Strong'},
  'liquidity': {'cash_and_equiv_usd': np.float64(9556000000.0),
                'current_rati

In [57]:
combined_list = []

# Create a dictionary mapping ticker -> ticker metrics dict
ticker_dict = {item['ticker']: item for item in ticker_metric_list}

# For each financial metrics dict, merge with the corresponding ticker metrics
for fin_item in financial_metric_list:
    ticker = fin_item['ticker']
    if ticker in ticker_dict:
        merged = {**ticker_dict[ticker], **fin_item}  # ticker metrics first, then financial (overwrites if needed)
        combined_list.append(merged)
    else:
        # If a ticker appears only in financial list (unlikely here), still include it
        combined_list.append(fin_item)

pprint.pprint(combined_list)

[{'cash_flow_health': {'fcf_margin': np.float64(0.16662711783968487),
                       'fcf_margin_rating': 'Excellent',
                       'free_cash_flow_usd': np.float64(14044000000.0),
                       'ocf_to_debt': np.float64(0.7128225645129026),
                       'ocf_to_debt_rating': 'Very Strong'},
  'debt_capacity': {'additional_debt_capacity_usd': None,
                    'assumptions': 'Maintain interest coverage >= 3.0x'},
  'growth_indicators': {'ebitda_growth_yoy': np.float64(0.059295013727747765),
                        'revenue_growth_yoy': np.float64(0.0029153131284284676)},
  'innovation_decay_slope': None,
  'innovation_freq': {},
  'latest_year': 2025,
  'leverage_tolerance': {'debt_ebitda_rating': 'Moderate',
                         'debt_to_ebitda': np.float64(1.0448977885539903),
                         'interest_coverage': np.float64(23.2348401323043),
                         'rating': 'Very Strong'},
  'liquidity': {'cash_and_equiv_us

In [ ]:
# Combine the financial and stagnation metric lists by ticker
ticker_dict = {item['ticker']: item for item in ticker_metric_list}
combined_list = []

for fin_item in financial_metric_list:
    ticker = fin_item['ticker']
    if ticker in ticker_dict:
        merged = {**ticker_dict[ticker], **fin_item}
        combined_list.append(merged)
    else:
        combined_list.append(fin_item)

# Add any tickers only in ticker_metric_list (safety)
for ticker, metric_item in ticker_dict.items():
    if not any(fin['ticker'] == ticker for fin in financial_metric_list):
        combined_list.append(metric_item)

# Convert to pandas DataFrame with ticker as primary key 
df = pd.DataFrame(combined_list)
df.set_index('ticker', inplace=True)
df.head()

,innovation_freq,innovation_decay_slope,topic_rigidity,topic_similarities,strategic_freq,strategic_decay_slope,latest_year,leverage_tolerance,debt_capacity,cash_flow_health,liquidity,growth_indicators,overall_feasibility,sentiment_by_year,capex_growth,sentiment_growth_correlation
ticker,,,,,,,,,,,,,,,,
PG,{},NaN,NaN,[],{},NaN,2025,"{'interest_coverage': 23.2348401323043, 'ratin...","{'additional_debt_capacity_usd': None, 'assump...","{'free_cash_flow_usd': 14044000000.0, 'fcf_mar...","{'current_ratio': 0.7041987908369849, 'rating'...","{'revenue_growth_yoy': 0.0029153131284284676, ...",Feasible – strong financial health,NaN,NaN,NaN
PGR,"{2023: 0.0, 2024: 0.0, 2025: 0.0, 2026: 0.0}",0.000000,0.999077,"[0.99910337, 0.9989146, 0.999212]","{2023: 0.0, 2024: 0.0, 2025: 0.0, 2026: 0.0}",0.000000,2025,"{'interest_coverage': 52.16187050359712, 'rati...","{'additional_debt_capacity_usd': None, 'assump...","{'free_cash_flow_usd': 17200000000.0, 'fcf_mar...","{'current_ratio': None, 'rating': 'N/A', 'cash...","{'revenue_growth_yoy': 0.16317375204066734, 'e...",Marginally feasible – monitor key risks,"{2023: 0.5, 2024: 0.5, 2025: 0.5}","{2024: -0.13095238095238096, 2025: -0.22105263...",NaN
PLD,"{2023: 0.006786507776206827, 2024: 0.005984958...",-0.000241,0.989795,"[0.9854278, 0.9985825, 0.9853747]","{2023: 0.014259745505958392, 2024: 0.013859904...",0.000090,2025,"{'interest_coverage': 4.876864086627123, 'rati...","{'additional_debt_capacity_usd': None, 'assump...","{'free_cash_flow_usd': None, 'fcf_margin': Non...","{'current_ratio': 0.9085606225907339, 'rating'...","{'revenue_growth_yoy': 0.07175627712119938, 'e...",Challenging – improvement needed,NaN,NaN,NaN
PRU,"{2023: 0.005972143493400276, 2024: 0.009545945...",0.000643,0.820899,"[0.46795827, 0.99503183, 0.9997059]","{2023: 0.006437768240343348, 2024: 0.004814476...",0.000205,2025,"{'interest_coverage': None, 'rating': 'N/A', '...","{'additional_debt_capacity_usd': None, 'assump...","{'free_cash_flow_usd': None, 'fcf_margin': Non...","{'current_ratio': None, 'rating': 'N/A', 'cash...","{'revenue_growth_yoy': -0.13736116024053768, '...",Infeasible – high risk of distress,NaN,NaN,NaN
PEG,"{2023: 0.007272628224334704, 2024: 0.007776521...",0.000351,0.961526,"[0.98511446, 1.0, 0.8994629]","{2023: 0.00332307357066979, 2024: 0.0036124975...",0.000183,2025,"{'interest_coverage': 3.470343392299688, 'rati...","{'additional_debt_capacity_usd': None, 'assump...","{'free_cash_flow_usd': 26000000.0, 'fcf_margin...","{'current_ratio': 0.8006968641114982, 'rating'...","{'revenue_growth_yoy': 0.1825072886297376, 'eb...",Challenging – improvement needed,"{2023: 0.5573399722828375, 2024: 0.56128659108...","{2024: -0.016541353383458645, 2025: 0.03195266...",1.0


In [ ]:
# Remove duplicate tickers 
result_df = df.drop_duplicates(subset=['ticker'], keep='first')

def is_empty_for_removal(val):
    """Return True if val is considered empty for removal rules."""
    # NaN or None
    if pd.isna(val):
        return True
    # Empty dict or string "{}"
    if isinstance(val, dict) and not val:
        return True
    if isinstance(val, str) and val.strip() == "{}":
        return True
    return False

cols_to_check = [
    'innovation_freq',
    'innovation_decay_slope',
    'topic_rigidity',
    'sentiment_growth_correlation'
]

mask_all_empty = (
    result_df['innovation_freq'].apply(is_empty_for_removal) &
    result_df['innovation_decay_slope'].apply(is_empty_for_removal) &
    result_df['topic_rigidity'].apply(is_empty_for_removal) &
    result_df['sentiment_growth_correlation'].apply(is_empty_for_removal)
)

result_df = result_df[~mask_all_empty]

# reset index for cleanliness
result_df = result_df.reset_index(drop=True)

In [ ]:
df.to_csv('sp500_stagnation_metric_df.csv')